# SARScope — data cleaning and modeling walkthrough

North Lake Inba, Japan. This notebook explains and reruns the completed threshold-baseline pipeline. No supervised ML model is claimed. Run cells from top to bottom after installing the project requirements. Saved figures and metrics are available even without running this notebook.

**Latest evaluation status:** See `reports/VALIDATION_RESULTS.md`. The frozen model passed its integrity check. August temporal evaluation is inconclusive; a separate northern-area check has F1 0.789 on only 0.5564 km² of usable data. The notebook below walks through the original baseline; those results are separate from later ML and validation experiments.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'backend/src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'backend/src').exists(), 'Open from the project or notebooks directory'
sys.path.insert(0, str(ROOT / 'backend'))
from IPython.display import display, Image, Markdown


## 1. Inputs and quality inspection
Three GCOV dates, identical 20 m grids, HH/HV intensities, and a GSI reference dated one day before the event observation. The crop is reference-informed, not a blind regional evaluation.

In [ ]:
inspection = json.loads((ROOT / 'data/processed/inspection_report.json').read_text())
print(json.dumps(inspection, indent=2))

## 2. Cleaning
Average a 5×5 neighborhood in linear power, requiring at least 20 valid neighbors. Missing center pixels remain missing. This is simple speckle reduction and blurs boundaries; the 20 m pixel spacing does not imply 20 m effective detail after filtering.

In [ ]:
from src.clean_data import run as clean
clean()
print((ROOT / 'reports/cleaning.json').read_text())

## 3. Baseline model
Otsu estimates an HH intensity gate from the pooled radar observations. New-water candidates must cross from bright to dark, decrease by at least 3 dB, and form groups of at least 9 pixels. No reference labels enter threshold fitting. HV is prepared but not used by this model.

In [ ]:
from src.model import run as model
parameters = model()
print(json.dumps(parameters, indent=2))

## 4. Reference comparison
This is descriptive spatial agreement. Outside-reference pixels are not verified dry, and the dates differ. Do not call the overlap fraction accuracy.

In [ ]:
from src.evaluate import run as evaluate
results = evaluate()
print(json.dumps(results, indent=2))

## 5. Show the result
The saved baseline identified 0.626 km² of candidate new open water. 83.9% of that area overlaps the reference, but only 16.6% of the valid reference area is captured. This substantial omission must be reported.

In [ ]:
from src.visualize_results import run as plot
plot()
display(Image(filename=str(ROOT / 'outputs/candidate_water_comparison.png')))

## 6. Threshold sensitivity
Nine settings are shown without choosing a winner. Future threshold tuning makes this event development data; use a separate event or reserved blocks for final evaluation.

In [ ]:
import csv
with (ROOT / 'reports/threshold_sensitivity.csv').open() as f:
    for row in csv.DictReader(f):
        print(row)

## 7. Implementation tests

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=ROOT / 'backend', check=True)

## 8. What is needed for supervised ML?
Independent water/nonwater labels, terrain and permanent-water context, explicit spatial/event holdouts, and comparison to this baseline. Do not present a classifier trained on this detector’s pseudo-labels as independently validated.

See `reports/PROJECT_REPORT.md` for the complete method, limitations, and data attribution.

## 9. Supervised Random Forest: next experiment
The training code is implemented in `src/train_ml.py`, but the real model is not trained until the six NASA OPERA TIFFs in `DOWNLOAD_NEXT.md` are available. From the project directory run `python -m src.train_ml`. This uses spatially separated blocks, ignores uncertain labels, and reports proxy-label agreement. It does not use GSI as training truth. See README for details and external-drive paths.

## Latest result: real Random Forest trained
See `reports/ML_RESULTS.md`: October 8 proxy labels supported training. Held-out spatial water F1 is 0.478, and temporal transfer is unvalidated. The preceding threshold-baseline cells describe a separate experiment. The missing-label note above was the prior status; the new explicit experiment is `configs/ml_october.json`.